# Why $\hat S$ is not unitary

This notebook checks the truncated shift used in the Dirichlet heat-equation discussion,

$$
\hat S = \sum_{x=0}^{M-2} |x+1\rangle\langle x|.
$$

On a finite computational basis, this operator is **not** unitary because it maps the last basis state to zero rather than wrapping it back to $|0\rangle$. We also compare it with the modular shift, which *is* unitary but corresponds to periodic boundary conditions instead of Dirichlet ones.

In [6]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator


def truncated_shift(M: int) -> np.ndarray:
    """Dirichlet-style shift: |x> -> |x+1> for x <= M-2, and |M-1> -> 0."""
    S = np.zeros((M, M), dtype=complex)
    for x in range(M - 1):
        S[x + 1, x] = 1.0
    return S


def modular_shift(M: int) -> np.ndarray:
    """Periodic shift: |x> -> |x+1 mod M>."""
    S = np.zeros((M, M), dtype=complex)
    for x in range(M):
        S[(x + 1) % M, x] = 1.0
    return S


def is_unitary(U: np.ndarray, atol: float = 1e-12) -> bool:
    I = np.eye(U.shape[0], dtype=complex)
    return np.allclose(U.conj().T @ U, I, atol=atol) and np.allclose(U @ U.conj().T, I, atol=atol)


def mcx_incrementer(num_qubits: int) -> QuantumCircuit:
    """MCX-based modular incrementer with q_0 treated as the least significant bit."""
    qc = QuantumCircuit(num_qubits)
    for target in range(num_qubits - 1, 0, -1):
        qc.mcx(list(range(target)), target)
    qc.x(0)
    return qc


In [7]:
M = 4
I = np.eye(M, dtype=complex)

S = truncated_shift(M)
S_dagger = S.conj().T
S_mod = modular_shift(M)
S_mod_dagger = S_mod.conj().T

T_dirichlet = 2 * I - (S + S_dagger)
T_periodic = 2 * I - (S_mod + S_mod_dagger)

def basis_state(M: int, x: int) -> np.ndarray:
    e = np.zeros(M, dtype=complex)
    e[x] = 1.0
    return e

np.set_printoptions(suppress=True)

print("Truncated Dirichlet-style shift S:")
print(S)
print()

print("Adjoint S^dagger:")
print(S_dagger)
print()

print("Dirichlet Laplacian 2I - (S + S^dagger):")
print(T_dirichlet)
print()

print("S^dagger S:")
print(S_dagger @ S)
print()

print("S S^dagger:")
print(S @ S_dagger)
print()

print("Is truncated S unitary?", is_unitary(S))
print()

print("S |M-1> =")
print(S @ basis_state(M, M - 1))
print()

print("Action of S on basis states:")
for x in range(M):
    image = S @ basis_state(M, x)
    print(f"S |{x}> =", image)
print()

print("Modular shift S_mod:")
print(S_mod)
print()

print("Action of S_mod on basis states:")
for x in range(M):
    image = S_mod @ basis_state(M, x)
    target = int(np.argmax(np.abs(image)))
    print(f"S_mod |{x}> = |{target}> (mod {M})")
print()

print("Periodic Laplacian 2I - (S_mod + S_mod^dagger):")
print(T_periodic)
print()

print("Is modular shift unitary?", is_unitary(S_mod))
print()

print("Interpretation:")
print("- The Dirichlet truncated shift is not unitary because it annihilates |M-1>.")
print("- Its Laplacian 2I - (S + S^dagger) is tridiagonal.")
print("- The modular shift satisfies S_mod |x> = |x+1 mod M>.")
print("- Its matrix has a wrap-around corner entry, which is exactly what makes the boundary condition periodic.")

Truncated Dirichlet-style shift S:
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]]

Adjoint S^dagger:
[[0.-0.j 1.-0.j 0.-0.j 0.-0.j]
 [0.-0.j 0.-0.j 1.-0.j 0.-0.j]
 [0.-0.j 0.-0.j 0.-0.j 1.-0.j]
 [0.-0.j 0.-0.j 0.-0.j 0.-0.j]]

Dirichlet Laplacian 2I - (S + S^dagger):
[[ 2.+0.j -1.+0.j  0.+0.j  0.+0.j]
 [-1.+0.j  2.+0.j -1.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  2.+0.j -1.+0.j]
 [ 0.+0.j  0.+0.j -1.+0.j  2.+0.j]]

S^dagger S:
[[1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]

S S^dagger:
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]

Is truncated S unitary? False

S |M-1> =
[0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Action of S on basis states:
S |0> = [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
S |1> = [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
S |2> = [0.+0.j 0.+0.j 0.+0.j 1.+0.j]
S |3> = [0.+0.j 0.+0.

In [8]:
S_mod.real

array([[0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.]])

In [9]:
num_qubits = 3
qc = mcx_incrementer(num_qubits)
U_qiskit = Operator(qc).data
S_mod_qiskit = modular_shift(2**num_qubits)

print("Qiskit MCX incrementer circuit (with q_0 as the least significant bit):")
print(qc.draw("text"))
print()

print("Operator matrix from Qiskit:")
print(np.real_if_close(U_qiskit))
print()

print("Analytic modular shift:")
print(np.real_if_close(S_mod_qiskit))
print()

print("Do the two matrices match?", np.allclose(U_qiskit, S_mod_qiskit))


Qiskit MCX incrementer circuit (with q_0 as the least significant bit):
               ┌───┐
q_0: ──■────■──┤ X ├
       │  ┌─┴─┐└───┘
q_1: ──■──┤ X ├─────
     ┌─┴─┐└───┘     
q_2: ┤ X ├──────────
     └───┘          

Operator matrix from Qiskit:
[[0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]]

Analytic modular shift:
[[0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]]

Do the two matrices match? True
